In [1]:
!pip install ollama langchain langgraph langchain_ollama pydantic langchain_community dotenv
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Reading package lists... Done
Building dependency tree... Done
Rea

In [2]:
!nohup ollama serve > ollama.log 2>&1 &
import time
# Give ollama serve some time to initialize
time.sleep(5)
print("Attempting to pull granite4.1:8b model...")
!ollama pull granite4.1:8bl

import ollama
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.1")
print("Ollama and granite4.1:8b model are ready.")

Attempting to pull granite4.1:8b model...

Error: pull model manifest: file does not exist
Ollama and granite4.1:8b model are ready.


In [ ]:
from backend.src.grpah.state import *
from backend.services import videoIndexingService
import logging
import os

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

In [ ]:
def video_indexer_node(state:VideoState) -> Dict[str,any]:
    """"
    Downloads the yt video from url 
    upload to gcp video indexer
    extracts the insights
    """
    url = state['video_url']
    id = state['video_id']

    logger.info(f"[Node:Indexer] Processing {url}")

    local_filename = "temp_audit_video.mp4"

    try:
        vi_service = videoIndexingService()
        # download   
        if "youtue.com" in url or "youtu.be" in url:
            local_path = vi_service.download_video(url,ouptput_path=local_filename)
        else:
            raise Exception("provide a valid yt video link")
        gcp_video_id = vi_service.upload_video(local_path,video_name=id)
        logger.info(f"upload success gcp id {gcp_video_id}")

        if os.path.exists(local_path):
            os.remove(local_path)
        raw_insights = vi_service.wait_for_processing(gcp_video_id)
        clean_data = vi_service.extract_data(raw_insights)
        logger.info(f"Extraction complete")
        return clean_data
    except Exception as e:
        logger.error(f"video indexer falied: {e}")
        return {
            "final_status": "FAIL",
            "transcript":"",
            "ocr_txt": []
        }
    

